In [2]:
import numpy as np
from tdmclient import ClientAsync
import cv2

from src.local_navigation.navigation_control import *
from src.computer_vision.vision import Vision 
from src.global_navigation.global_nav import global_nav
from src.pose_estimation.extended_kalman_filter import ExtendedKalmanFilter
from src.pose_estimation.differential_drive_system import DifferentialDriveSystem
# Progress bar for loops visualization
from tqdm import tqdm
# Plotting
import matplotlib.pyplot as plt
from src.utils.plot_pose_estimation import *
%matplotlib inline


In [ ]:
import cv2

# Try with 0 first – that's usually the default camera
cap = cv2.VideoCapture(1)  # change to 1, 2... if needed

if not cap.isOpened():
    print("❌ Impossible d'ouvrir la webcam.")
    exit(1)

print("✅ Webcam ouverte, appuyez sur 'q' pour quitter.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Impossible de capturer une frame.")
        break

    cv2.imshow("Test webcam", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import time

vision = Vision()

# 1) Open the webcam and capture ONE reference image
cap = cv2.VideoCapture(1)  # use index 1 because this is the one that works for you

if not cap.isOpened():
    raise RuntimeError("Failed to open webcam (index 1).")

time.sleep(1)  # wait for the camera to stabilize
ret, frame0 = cap.read()
if not ret or frame0 is None:
    cap.release()
    raise RuntimeError("Failed to capture a frame from the webcam.")

# We can close the camera now; the reference image is stored in frame0
cap.release()

# Just to verify that we really have an image
print("[DEBUG] Captured frame shape:", frame0.shape)
cv2.imshow("Reference frame", frame0)
cv2.waitKey(1)  # small delay so the window can appear at least once

# 2) Initialize your Vision pipeline with the reference frame
print("[DEBUG] Calling vision.initialize(...)")
vision.initialize(frame0)  # e.g., color detection, blur, Canny, polygons, etc.

# 3) Process the same frame to extract obstacles and robot state
print("[DEBUG] Calling vision.process(...)")
obstacles, robot_state = vision.process(
    frame0,
    show_debug=False,    # no debug display from inside Vision
    skip_blur=False,
    log_pose=False
)

print("[DEBUG] Robot state:", robot_state)

# 4) Global navigation using the detected obstacles
robot_radius = 30
imgh, imgw = frame0.shape[:2]

print("[DEBUG] Calling global_nav(...)")
path = global_nav(obstacles, robot_radius, imgh, imgw, frame0)

print("[DEBUG] global_nav finished, path length:",
      len(path) if path is not None else None)

# 5) Keep the window open until the user presses 'q' or 'Esc'
while True:
    # If you want to keep showing the reference frame
    cv2.imshow("Reference frame", frame0)

    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):
        break

cv2.destroyAllWindows()


[DEBUG] Captured frame shape: (480, 640, 3)
[DEBUG] Calling vision.initialize(...)
[INFO] Initialisation terminé ( couleurs + Canny + polygones figé).
[DEBUG] Calling vision.process(...)
[DEBUG] Robot state: {'found': False, 'center': None, 'theta': None, 'red_center': None, 'green_center': None, 'robot_mask': None, 'smooth_center': None, 'smooth_theta': None}
[DEBUG] Calling global_nav(...)


ValueError: A linearring requires at least 4 coordinates.

: 

In [3]:
vision = Vision()
#img = vision.load("initial_state.png")

        # 1) Ouvre la caméra et capture UNE image de référence
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    raise RuntimeError("Impossible d'ouvrir la webcam.")

ret, frame0 = cap.read()
if not ret:
    cap.release()
    raise RuntimeError("Impossible de capturer une frame webcam.")

cap.release()  # on ferme direct, l’image est stockée dans frame0

vision.initialize(frame0)  # couleurs + blur robot + canny + polygones (et fige dans static_polys)

# récupère les obstacles figés par initialize()
obstacles, robot_state = vision.process(
    frame0,
    show_debug=False,    # on ne veut pas l'image debug ici
    skip_blur=False,
    log_pose=False
)

print(robot_state)
robot_radius = 30
imgh, imgw = frame0.shape[:2]
path= global_nav(obstacles, robot_radius,imgh, imgw, frame0)


while True:
        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord('q'):
            break

RuntimeError: Impossible de capturer une frame webcam.

In [ ]:
# 1.6 EKF + Thymio client are created elsewhere, e.g.
# ekf = YourEKF(...)
# thymio = YourThymioClient(...)
lambda_ = 0.39735099337748336  # conversion factor
d = 93.5         # distance between wheels
# Default value for now
Q = np.diag([0.03, 0.03, 0.03])
R = np.diag([0.01, 0.01, 0.01])
dt = 0.1  # time step

system = DifferentialDriveSystem(dt, lambda_, d, Q, R)
    
# Initial state and covariance
mu0 = np.array([0, 0, 0])
Sigma0 = 1000 * np.eye(3)

ekf = ExtendedKalmanFilter(
    mu0, Sigma0,
    system
)

# To store estimates
x_est = []
P_est = []

In [ ]:
async def run():
    client = ClientAsync()
    node = await client.wait_for_node()
    await node.lock()
    await node.wait_for_variables({"prox.horizontal"})


    print("Starting path following with obstacle avoidance...")
    for step in range(200):
        # Get camera image and process
        ret, frame = cap.read()
        if not ret:
            cap.release()
            raise RuntimeError("Impossible de capturer une frame webcam.")
        
        robot_state = vision.process(
                                frame,
                                show_debug=False,    # on ne veut pas l'image debug ici
                                skip_blur=False,
                                log_pose=False
                            )

        # --- get sensors from Thymio ---
        sensor_vals = list(node.v.prox.horizontal)
        
        speed_left  = node.v.motor.left.speed
        speed_right = node.v.motor.right.speed
        u = np.array([speed_left, speed_right])
        # pose from ekf 
        pose, cov = ekf.predict(u)

        if robot_state["found"]:
            x, y = robot_state["center"]
            theta = robot_state["angle"]
            z = np.array([x, y, theta])
            pose , cov = ekf.update(z)

        x_est.append(pose)
        P_est.append(cov)

        # --- update local grid ---
        grid.update_from_sensor_vals(sensor_vals)

        # --- compute virtual goal ---
        virt_world, LA_world, F_att, F_rep = navigator.compute_virtual_goal(
            pose, global_path, sensor_vals=sensor_vals
        )
        if virt_world is None:
            break

        # --- controller: pose -> motors ---
        uL, uR, info = g2g.compute_motor_commands(pose, virt_world)

        await node.set_variables({
            "motor.left.target":  [uL],
            "motor.right.target": [uR],
        })

        
        # small sleep
        await client.sleep(dt)

    # stop motors at the end
    await node.set_variables({
        "motor.left.target":  [0],
        "motor.right.target": [0],
    })
    await node.unlock()